# Lean 4 Theorem Proving with Qwen3-4B + LoRA

**Model**: [`yotsubian/qwen`](https://huggingface.co/yotsubian/qwen) — Qwen3-4B fine-tuned on Lean 4 proofs  
**Problem**: `mathd_algebra_141` from miniF2F — *If 2(a+b)=30 and a+2b=26, prove a=4*

In [1]:
import re, torch, json
from transformers import AutoTokenizer, AutoModelForCausalLM
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER    = "yotsubian/qwen"

LEAN4_HEADER = (
    "import Mathlib\nimport Aesop\n\n"
    "set_option maxHeartbeats 0\n\n"
    "open BigOperators Real Nat Topology Rat\n\n"
)

STATEMENT = (
    "theorem mathd_algebra_141 (a b : \u211d) "
    "(h\u2080 : 2 * (a + b) = 30) (h\u2081 : a + 2 * b = 26) : a = 4 := by"
)

In [2]:
# Download adapter files from HuggingFace
print("Downloading adapter...")
adapter_dir = snapshot_download(ADAPTER)

with open(f"{adapter_dir}/adapter_config.json") as f:
    cfg = json.load(f)
scaling = cfg["lora_alpha"] / cfg["r"]
print(f"LoRA r={cfg['r']}, alpha={cfg['lora_alpha']}, scaling={scaling}")

lora_weights = load_file(f"{adapter_dir}/adapter_model.safetensors")
print(f"Adapter keys: {len(lora_weights)}")

Fetching 49 files:   0%|          | 0/49 [00:00<?, ?it/s]

LoRA r=16, alpha=32, scaling=2.0
Adapter keys: 288


In [3]:
print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype).to("cpu")
state = model.state_dict()
print("Base model loaded.")

Loading base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Build param lookup by name
params = dict(model.named_parameters())

merged = 0
for key, B in lora_weights.items():
    if ".lora_B." not in key:
        continue
    A_key = key.replace(".lora_B.", ".lora_A.")
    A     = lora_weights[A_key]

    # peft key: base_model.model.<path>.lora_B.weight
    # param name:                <path>.weight
    base_key = key
    if base_key.startswith("base_model.model."):
        base_key = base_key[len("base_model.model."):]
    base_key = re.sub(r"\.lora_[AB]\.weight$", ".weight", base_key)

    if base_key not in params:
        print(f"WARNING: no match for {key!r} -> {base_key!r}")
        continue

    delta = (B @ A).to(params[base_key].dtype)
    with torch.no_grad():
        params[base_key].add_(scaling * delta)
    merged += 1

model.eval()
print(f"Merged {merged} LoRA pairs.")

Merged 144 LoRA pairs.


In [ ]:
user_msg = (
    "Complete the following Lean 4 code:\n\n"
    "```lean4\n" + LEAN4_HEADER + STATEMENT + "\n```"
)
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_msg}],
    tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
in_len = inputs["input_ids"].shape[1]

print("Generating proof...")
with torch.no_grad():
    out = model.generate(
        **inputs, max_new_tokens=512,
        do_sample=True, temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )

raw = tokenizer.decode(out[0][in_len:], skip_special_tokens=True)
print(raw)

Generating proof...


RuntimeError: Tensor.item() cannot be called on meta tensors

In [ ]:
def extract_proof(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    if text.startswith("```lean4"): text = text[8:].strip()
    elif text.startswith("```"):    text = text[3:].strip()
    if text.endswith("```"):        text = text[:-3].strip()
    return text

proof     = extract_proof(raw)
lean_file = LEAN4_HEADER + STATEMENT + "\n" + proof
print(lean_file)

In [ ]:
try:
    import requests
    r = requests.post("http://localhost:8000", json={
        "cmd": lean_file, "allTactics": False, "ast": False,
        "tactics": False, "premises": False,
    }, timeout=60)
    data = r.json()
    errors = [m for m in data.get("messages", []) if m["severity"] == "error"]
    if not errors and not data.get("sorries"):
        print("PROOF VERIFIED:", data)
    else:
        for e in errors: print("Error:", e["data"][:120])
except Exception:
    print("Lean server not running — start with:")
    print("  python lean_server.py --workspace /path/to/mathlib4 --port 8000")